# NB 4 — Memory, and the stale-fact trap
**Goal:** show that memory is a *design choice*, not a free feature — and that memory which does not track freshness will **resurface a stale fact and drive a confidently wrong action.**

An agent is only as good as the state it remembers. Here the same agent, on the same patient, gives a safe or an unsafe recommendation depending purely on *how* its long-term memory is recalled. (Runs in MOCK mode with no API key.)

In [1]:
import os, json, re

# =============================================================
# Model backend — works two ways:
#   1) MOCK (default): no API key needed. Returns scripted responses
#      so you can run the whole notebook and see the STRUCTURE.
#   2) REAL model: pip install openai, then either
#        - Cloud:  export OPENAI_API_KEY=sk-...        (uses OpenAI)
#        - Local open-weight (vLLM / LM Studio / Ollama):
#              export OPENAI_BASE_URL=http://localhost:8000/v1
#              export OPENAI_API_KEY=dummy
#              export MODEL=meta-llama/Llama-3.1-8B-Instruct   # your served model
# Everything below is model-agnostic: swap the model, keep the code.
# =============================================================
USE_MOCK = os.environ.get("OPENAI_API_KEY") is None
MODEL    = os.environ.get("MODEL", "gpt-4o-mini")

def chat(messages, temperature=0):
    """Return the assistant's text for a list of {role, content} messages."""
    if USE_MOCK:
        return _mock(messages, temperature)
    from openai import OpenAI
    client = OpenAI(base_url=os.environ.get("OPENAI_BASE_URL"))  # None -> api.openai.com
    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature, timeout=60)
    return r.choices[0].message.content

print("Backend:", "MOCK (no key found — scripted demo)" if USE_MOCK else f"REAL model = {MODEL}")

def _mock(messages, temperature=0):
    """Scripted clinician-assistant reply. The recommendation depends ENTIRELY on
    which memory the agent was given: a stale 'still on warfarin' fact produces a
    confidently wrong (and unsafe) recommendation; the current fact produces the
    right one. A real model behaves the same way — it can only reason over what
    memory hands it."""
    u = " ".join(m["content"] for m in messages if m["role"] == "user")
    if "DISCONTINUED" in u:
        return ("Warfarin was discontinued, so the elevated INR is not warfarin-related: do NOT "
                "adjust or restart warfarin. Flag for clinician review and work up other causes "
                "(liver function, nutrition, interacting meds).")
    return ("INR 4.2 is above the 2.0-3.0 range on warfarin: hold the next warfarin dose and "
            "recheck INR in 2-3 days.")

Backend: REAL model = openai/gpt-4o-mini


### Two kinds of memory
- **Short-term** memory is the context window: what's in front of the model right now.
- **Long-term** memory is an external store the agent writes to and reads back over time.

The store below keeps every fact with the **day** it was recorded, so we can compare two recall strategies later.

In [2]:
# Long-term memory: an append-only log of facts, each stamped with the day it was written.
MEMORY = []
def remember(kind, value, day):
    MEMORY.append({"kind": kind, "value": value, "day": day})

# Two ways to recall a fact of a given kind:
def recall_naive(kind):
    "First thing we ever wrote — no notion of freshness. (A very common bug.)"
    return next((f for f in MEMORY if f["kind"] == kind), None)

def recall_current(kind):
    "Newest fact of that kind wins — memory that respects time."
    facts = [f for f in MEMORY if f["kind"] == kind]
    return max(facts, key=lambda f: f["day"]) if facts else None

### A timeline where a fact goes stale
The patient starts warfarin, then it is **discontinued** three weeks later. Both events are recorded. Today (day 30) a new INR comes back high.

In [3]:
remember("anticoagulant", "warfarin 5 mg daily", day=1)                       # started
remember("anticoagulant", "warfarin DISCONTINUED (day 20, bleeding risk)", day=20)  # stopped
TODAY_INR = 4.2   # above the 2.0-3.0 range

for f in MEMORY:
    print(f'  day {f["day"]:>2}: {f["value"]}')

  day  1: warfarin 5 mg daily
  day 20: warfarin DISCONTINUED (day 20, bleeding risk)


### The agent drafts a recommendation from whatever memory it is handed

In [4]:
def draft_recommendation(recalled):
    msgs = [{"role":"system","content":"You are a clinical follow-up assistant. Given the anticoagulant "
             "status from memory and the latest INR, recommend the next step in one or two sentences. "
             "Do not change any order yourself."},
            {"role":"user","content":f"Anticoagulant status (from memory): {recalled['value']}. "
             f"Latest INR: {TODAY_INR} (range 2.0-3.0). What is the next step?"}]
    return chat(msgs)

print("NAIVE recall  ->", recall_naive("anticoagulant")["value"])
print("  recommendation:", draft_recommendation(recall_naive("anticoagulant")))
print()
print("CURRENT recall ->", recall_current("anticoagulant")["value"])
print("  recommendation:", draft_recommendation(recall_current("anticoagulant")))

NAIVE recall  -> warfarin 5 mg daily


  recommendation: Hold the warfarin dose and consider administering oral vitamin K, especially if the INR is above 4.0 and there is a risk of bleeding. Monitor the INR closely and reassess in 1-2 days.

CURRENT recall -> warfarin DISCONTINUED (day 20, bleeding risk)


  recommendation: Given that warfarin has been discontinued and the INR is elevated at 4.2, it is recommended to monitor the INR closely and consider administering oral vitamin K if the INR remains above 4.0 or if there are signs of bleeding.


### Takeaway
Both runs used the **same model and the same patient** — only the memory recall differed. Naive recall handed the agent a fact that was true three weeks ago and is now dangerous, and the agent had no way to know. Durable memory is what lets agents work over time, but *stale memory is a first-class failure mode*: recall has to respect recency, supersession, and provenance. In a real system these facts would carry timestamps and status from the record (Synthea / FHIR), and "memory hygiene" — expiring, superseding, and re-verifying — is part of the design, not an afterthought.

*Try:* point the backend at a real open-weight model and re-run — the naive-recall recommendation stays wrong, because the problem is the memory, not the model.